### Lock, Event, Condition

In [ ]:
import asyncio
import time

start_time = time.monotonic()

def log(message: str) -> None:
    elapsed = time.monotonic() - start_time
    print(f"[{elapsed:6.2f}s] {message}")

In [ ]:
async def make_item(item: str = "coffee", brew: float = 2.0, fail: bool = False) -> str:
    try:
        await asyncio.sleep(brew)
    except asyncio.CancelledError:
        log(f"{item}: thrown out")
        raise
    if fail:
        raise RuntimeError(f"out of {item}")
    return item

**Race condition — no lock**

In [ ]:
start_time = time.monotonic()

board = {"cups": 0}


async def serve_unsafe(board: dict):
    await make_item(brew=0)
    current = board["cups"]
    await asyncio.sleep(0)
    board["cups"] = current + 1


await asyncio.gather(*(serve_unsafe(board) for _ in range(100)))
log(f"no lock: {board['cups']} (expected 100)")

**With `Lock`**

In [ ]:
start_time = time.monotonic()

board = {"cups": 0}
board_lock = asyncio.Lock()


async def serve_safe(board_lock: asyncio.Lock, board: dict):
    await make_item(brew=0)
    async with board_lock:
        current = board["cups"]
        await asyncio.sleep(0)
        board["cups"] = current + 1


await asyncio.gather(*(serve_safe(board_lock, board) for _ in range(100)))
log(f"with lock: {board['cups']} (expected 100)")

**`Event` — customers wait, café opens, then order**

In [ ]:
start_time = time.monotonic()

doors_open = asyncio.Event()
print(doors_open.is_set())  # False


async def customer(doors_open: asyncio.Event, customer_name: str):
    log(f"{customer_name}: waiting outside  | event is_set={doors_open.is_set()}")
    await doors_open.wait() # function waits here until set() is called!
    log(f"{customer_name}: going in         | event is_set={doors_open.is_set()}")
    await make_item(f"{customer_name}'s order", brew=0.3)
    log(f"{customer_name}: served")


async def open_cafe(doors_open: asyncio.Event):
    log(f"café: getting ready...     | event is_set={doors_open.is_set()}")
    await asyncio.sleep(1.0)
    log(f"café: doors open!          | event is_set={doors_open.is_set()}  (before set)")
    doors_open.set()
    log(f"café: set() called         | event is_set={doors_open.is_set()}  (after set)")


customers = [asyncio.create_task(customer(doors_open, f"customer-{index}")) for index in range(4)]
cafe = asyncio.create_task(open_cafe(doors_open))

await asyncio.gather(cafe, *customers)

**`Condition` — wait until enough cups on the warming shelf**

After `notify_all()`, every waiter re-checks its predicate. Alice takes 3 of the 5 → Bob's `cups >= 2` is true immediately.

In [ ]:
start_time = time.monotonic()

shelf = {"cups": 0}
condition = asyncio.Condition()


async def customer(customer_name: str, wanted: int):
    async with condition:
        log(f"{customer_name}: wants {wanted}, waiting")
        await condition.wait_for(lambda: shelf["cups"] >= wanted)
        shelf["cups"] -= wanted
        log(f"{customer_name}: took {wanted}, {shelf['cups']} left")


async def barista(amount: int):
    for _ in range(amount):
        await make_item(brew=0.16)
    async with condition:
        shelf["cups"] += amount
        log(f"barista: +{amount}, shelf={shelf['cups']}")
        condition.notify_all()


customers = [
    asyncio.create_task(customer("Alice", 3)),
    asyncio.create_task(customer("Bob", 2)),
]
barista_task = asyncio.create_task(barista(5))

await asyncio.gather(barista_task, *customers)

---
- `Lock` makes shared-state updates atomic.
- `Event` is a one-shot start signal for many waiters.
- `Condition.wait_for(predicate)` parks until the predicate is true; pair with `notify_all()`.